<a href="https://colab.research.google.com/github/karttikjangid/Attention-is-All-you-need/blob/main/MicroGrad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
  return 3*x**2 -4*x + 5

In [ ]:
f(3)

In [ ]:
xs = np.arange(-5,5,0.25)
ys = f(xs)
plt.plot(xs,ys)

In [ ]:
h =  0.000001
x = 3.0
## derivative
(f(x+h) - f(x))/h

In [ ]:
h =  0.000001
x = -3.0
## derivative
(f(x+h) - f(x))/h

In [ ]:
h =  0.000001
x = 2/3
## derivative
(f(x+h) - f(x))/h

In [ ]:
h =  0.00001
#inputs
a = 2.0
b = -3.0
c = 10.0
d1 = a*b +c
a+=h
d2 = a*b +c
print('d1',d1)
print('d2',d2)
print('slope', (d2-d1)/h)


In [ ]:
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self._prev = set(_children)
    self._op = _op
    self.label = label
    self.grad = 0
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    return out

In [ ]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b
e.label = 'e'
d = e + c
d.label = 'd'
print(d)
print('Children of d:', d._prev)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        # The actual scalar number (e.g., 2.0 or -4.0)
        self.data = data

        # The "grad" is our "demon's blame factor" from Chapter 1.
        # It represents the derivative of the final output with respect to this value.
        # We initialize it to 0.0, assuming initially that it has no effect on the output.
        self.grad = 0.0

        # We keep a set of the variables that created this one (for the backward pass)
        self._prev = set(_children)

        # The mathematical operation that created this node (e.g., '+' or '*')
        self._op = _op

        # Just a string name for debugging/drawing the graph
        self.label = label
        self._backward = lambda : None

    def backward(self):
        # 1. Build the topological graph
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                # Recursively process all children first
                for child in v._prev:
                    build_topo(child)
                # Only add ourselves after all children are processed
                topo.append(v)

        # Start the sort from this current node (e.g., the final Loss)
        build_topo(self)

        # 2. Set the base case gradient to 1.0 (What you deduced in Chunk 2!)
        self.grad = 1.0

        # 3. Go backwards through the topological list and call _backward()
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data})"
    def __add__(self , other):
      if not isinstance(other, Value):
        other = Value(other)
      val = other.data
      out = Value(self.data + val , _children = (self , other) , op ='+');
      def _backward():
            # Chain rule: local derivative (1.0) * global derivative (out.grad)
            # Remember to ACCUMULATE (+=) to prevent the overwrite bug!
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        # Store the function in the output node
        out._backward = _backward
        return out
    def __mul__(self,other):
      if not isinstance(other, Value):
        other = Value(other)
      val = other.data
      out = Value(self.data * val , _children = (self , other) , op ='*');
      def _backward():
        ## Chain rule -> multiply by the other nodes data
        self.grad += other.data*out.grad
        other.grad += self.data*out.grad
        ## save it
      out._backward = _backward
      return out

    ## adding edge-cases (reverse multiplt)
    def __rmul__(self , other):
      return self*other
    def __radd__(self,other):
      return self+other

    def __pow__(self,other):
      out = Value(pow(self.data,other) , _children = (self,) ,_op ='**')

      def _backward():
        self.grad+= other*pow(self.data,other-1)*out.grad
      out._backward = _backward
      return out

    def __neg__(self):
      return self*-1
    def __sub__(self,other):
      neg_value = -other
      return self+neg_value
    def __truediv__(self,other):
      new_val = other**-1
      return self*new_val

    def tanh(self):
      t = (math.exp(2*self.data) -1) / (math.exp(2*self.data) + 1)
      out = Value(t , _children = (self,) , _op='tanh')

      def _backward():
        ## derivate of tanh(x) is 1- tanh(x)**2
        der = 1 - t**2
        self.grad += der*out.grad
      out._backward = _backward
      return out


In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        # nin is the number of inputs coming into this neuron.
        # We create a random weight (wrapped in our Value object) for every input.
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        # We create a single random bias.
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # x is a list of inputs
        out = sum([xi*wi for xi , wi in zip(x,self.w)]) + self.b
        return out.tanh()

    def parameters(self):
        return self.w + [self.b]


In [ ]:
class Layer:
    def __init__(self, nin, nout):
        # nin: number of inputs coming into the layer
        # nout: the number of neurons we want in this specific layer
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        # x is the list of inputs
        outputs = [n(x) for n in self.neurons]
        if len(outputs) == 1:
          return outputs[0]
        else :
          return outputs
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        # nin: the number of raw data inputs entering the network
        # nouts: a list of the sizes of every layer we want.
        # For example, nouts=[3, 4] means two hidden layers of 4, and an output layer of 1.

        # We combine the input size and all layer sizes into one list for easy iteration
        sz = [nin] + nouts

        # We create a list of Layer objects.
        # The input of a layer is the output of the previous one (sz[i]),
        # and its output is its current size (sz[i+1]).
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        # x is the initial list of raw inputs
        for i in range(len(self.layers)):
          x = self.layers[i](x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]




In [ ]:
for epoch in range(20):

    # 1. The Forward Pass: Get predictions for every input in our dataset
    y_pred = [n(x) for x in xs]

    # 2. Calculate the Loss (Mean Squared Error)
    # We pair up each prediction with its true target, subtract, square, and sum them up.
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, y_pred))

    # 3. Zero the Gradients
    # Crucial bug prevention: We must reset gradients to 0 before backpropagating [1].
    # Otherwise, gradients from the previous epoch will incorrectly accumulate [1].
    for p in n.parameters():
        p.grad = 0.0

    # 4. The Backward Pass: Trigger the "Chain of Blame"
    # This calculates the derivative of the loss with respect to every weight and bias.
    loss.backward()

    # 5. The Optimization Step (Gradient Descent)
    # We update every parameter by taking a small step (0.05) in the opposite
    # direction (-) of the gradient to minimize the loss [2].
    for p in n.parameters():
        p.data += -0.05 * p.grad

    # Print the progress
    print(f"Epoch {epoch} | Loss: {loss.data}")